In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
import pandas as pd
import os


import warnings
warnings.filterwarnings('ignore')


# Load your pickle file
# Load your pickle file
exp_name = '20241106_3'
pickle_path = '/run/user/1000/gvfs/smb-share:server=bucket.oist.jp,share=bucket/ReiterU/Anouk/basler_data/' + exp_name + '/'
#pickle_path = '/Volumes/ReiterU/Anouk/basler_data/' + exp_name + '/'
output_folder_path = '/run/user/1000/gvfs/smb-share:server=bucket.oist.jp,share=bucket/ReiterU/Anouk/basler_data/' + exp_name + '/'
#output_folder_path = '/Volumes/ReiterU/Anouk/basler_data/'+ exp_name + '/'


col_1, col_2 = 9, 11

pickle_name_col1 = exp_name + '_3_aruco_cleaned_with_avg_dist_col1.pkl'
pickle_name_col2 = exp_name + '_3_aruco_cleaned_with_avg_dist_col2.pkl'

In [2]:
def read_pickle_file_with_load(pickle_path, pickle_name):
    """
    Read a DataFrame from a pickle file using `pickle.load`, ensuring the file exists.

    Parameters:
    - pickle_path (str): Path to the folder where the pickle file is located.
    - pickle_name (str): Name of the pickle file (including '.pkl').

    Returns:
    - pd.DataFrame: Loaded DataFrame.
    """
    # Full path to the pickle file
    pickle_full_path = os.path.join(pickle_path, pickle_name)
    
    try:
        print(f"Reading DataFrame from {pickle_full_path} using `pickle.load`...")
        with open(pickle_full_path, 'rb') as f:
            data = pickle.load(f)
        print(f"File '{pickle_name}' read successfully.")
        return data
    except FileNotFoundError:
        print(f"Error: File '{pickle_full_path}' not found.")
    except Exception as e:
        print(f"Error while reading file: {e}")

In [ ]:
df_1 = read_pickle_file_with_load(output_folder_path,  pickle_name_col1)
df_2 = read_pickle_file_with_load(output_folder_path, pickle_name_col2)


Reading DataFrame from /run/user/1000/gvfs/smb-share:server=bucket.oist.jp,share=bucket/ReiterU/Anouk/basler_data/20241106_3/20241106_3_4_aruco_with_interactions_col1.pkl using `pickle.load`...


In [ ]:

# Define ROI (replace with your actual ROI conditions)
roi_bounds_nest_1 = {'x_min': -5300, 'x_max': -1600, 'y_min':-4000, 'y_max': -200}
roi_bounds_nest_2 = {'x_min': 5500, 'x_max': 9300, 'y_min':-4000, 'y_max': -300}



In [ ]:
#Colony 1 
# Determine if each point is inside or outside the ROI
df_1['inside_roi'] = (df_1['X'] >= roi_bounds_nest_1['x_min']) & (df_1['X'] <= roi_bounds_nest_1['x_max']) & \
                   (df_1['Y'] >= roi_bounds_nest_1['y_min']) & (df_1['Y'] <= roi_bounds_nest_1['y_max'])

# Scatter plot
plt.figure(figsize=(10, 8))
plt.scatter(df_1['X'], df_1['Y'], c=df_1['inside_roi'].map({True: 'blue', False: 'red'}), alpha=0.5, label='Points', s = 10)
plt.xlabel('X Position')
plt.ylabel('Y Position')
plt.title('Ant Positions with ROI Boundaries')

# Draw ROI boundary as a square
plt.plot(
    [roi_bounds_nest_1['x_min'], roi_bounds_nest_1['x_min'], roi_bounds_nest_1['x_max'], roi_bounds_nest_1['x_max'], roi_bounds_nest_1['x_min']],
    [roi_bounds_nest_1['y_min'], roi_bounds_nest_1['y_max'], roi_bounds_nest_1['y_max'], roi_bounds_nest_1['y_min'], roi_bounds_nest_1['y_min']],
    'g--', label='ROI Boundary'
)

# Add legend
plt.legend(['ROI Boundary', 'Inside ROI', 'Outside ROI'])
plt.show()


In [ ]:
#Colony 2
# Determine if each point is inside or outside the ROI
df_2['inside_roi'] = (df_2['X'] >= roi_bounds_nest_2['x_min']) & (df_2['X'] <= roi_bounds_nest_2['x_max']) & \
                   (df_2['Y'] >= roi_bounds_nest_2['y_min']) & (df_2['Y'] <= roi_bounds_nest_2['y_max'])

# Scatter plot
plt.figure(figsize=(10, 8))
plt.scatter(df_2['X'], df_2['Y'], c=df_2['inside_roi'].map({True: 'blue', False: 'red'}), alpha=0.5, label='Points', s = 10)
plt.xlabel('X Position')
plt.ylabel('Y Position')
plt.title('Ant Positions with ROI Boundaries')

# Draw ROI boundary as a square
plt.plot(
    [roi_bounds_nest_2['x_min'], roi_bounds_nest_2['x_min'], roi_bounds_nest_2['x_max'], roi_bounds_nest_2['x_max'], roi_bounds_nest_2['x_min']],
    [roi_bounds_nest_2['y_min'], roi_bounds_nest_2['y_max'], roi_bounds_nest_2['y_max'], roi_bounds_nest_2['y_min'], roi_bounds_nest_2['y_min']],
    'g--', label='ROI Boundary'
)

# Add legend
plt.legend(['ROI Boundary', 'Inside ROI', 'Outside ROI'])
plt.show()


In [ ]:

def calculate_time_distance_outside_roi(df, roi_bounds):
    """
    Calculates the time and distance each ARUCO marker spends outside a specified ROI.
    Adds 'time_outside_roi' and 'distance_outside_roi' columns to the dataframe.

    Parameters:
    - df (pd.DataFrame): DataFrame containing 'Frame_number', 'ARUCO_number', 'X', 'Y' columns.
    - roi_bounds (dict): Dictionary with 'x_min', 'x_max', 'y_min', 'y_max' for ROI bounds.
    
    Returns:
    - df (pd.DataFrame): Original DataFrame with added 'time_outside_roi' and 'distance_outside_roi' columns.
    """
    # Determine if each marker is outside the ROI
    df['outside_roi'] = ~((df['X'] >= roi_bounds['x_min']) & (df['X'] <= roi_bounds['x_max']) &
                          (df['Y'] >= roi_bounds['y_min']) & (df['Y'] <= roi_bounds['y_max']))

    # Group by ARUCO_number to calculate metrics per marker
    def calculate_metrics(group):
        # Calculate distance moved between consecutive frames
        group['distance'] = np.sqrt(group['X'].diff()**2 + group['Y'].diff()**2).fillna(0)
        
        # Only consider frames with non-zero distance for outside ROI calculations
        valid_frames = group['distance'] > 0
        
        # Time outside ROI is the count of valid frames outside ROI
        group['time_outside_roi'] = group['outside_roi'] & valid_frames
        group['time_outside_roi'] = group['time_outside_roi'].cumsum()
        
        # Cumulative distance outside ROI
        group['distance_outside_roi'] = group['distance'].where(group['outside_roi']).cumsum().fillna(0)
        
        # Reset metrics for markers that never leave the ROI
        if not group['outside_roi'].any():
            group['time_outside_roi'] = 0
            group['distance_outside_roi'] = 0
        
        return group

    # Apply the metrics calculation to each ARUCO marker group
    df = df.groupby('ARUCO_number').apply(calculate_metrics).reset_index(drop=True)

    return df

# Apply the function to your dataframes
df_1_with_metrics = calculate_time_distance_outside_roi(df_1, roi_bounds_nest_1)
df_2_with_metrics = calculate_time_distance_outside_roi(df_2, roi_bounds_nest_2)


In [ ]:
def plot_time_and_distance_per_aruco(df_with_metrics, col):
    time_outside_roi = df_with_metrics.groupby('ARUCO_number')['time_outside_roi'].max()
    distance_outside_roi = df_with_metrics.groupby('ARUCO_number')['distance_outside_roi'].max()
    
    fig, ax1 = plt.subplots(figsize=(15, 8))
    width = 0.35
    aruco_numbers = time_outside_roi.index
    x = np.arange(len(aruco_numbers))
    
    # Distance bar plot on the left y-axis
    ax1.bar(x - width/2, distance_outside_roi.values, width, color='purple', label='Distance Outside ROI')
    ax1.set_xlabel('ARUCO Number')
    ax1.set_ylabel('Distance Outside ROI', color='purple')
    ax1.tick_params(axis='y', labelcolor='purple')
    
    # Time bar plot on the right y-axis
    ax2 = ax1.twinx()
    ax2.bar(x + width/2, time_outside_roi.values, width, color='orange', label='Time Outside ROI')
    ax2.set_ylabel('Time Outside ROI (frames)', color='orange')
    ax2.tick_params(axis='y', labelcolor='orange')
    
    # x-axis settings
    ax1.set_xticks(x)
    ax1.set_xticklabels(aruco_numbers, rotation=90)
    ax1.set_title('Time and Distance Outside ROI for Each ARUCO in Colony ' + str(col))
    
    # Show plot
    fig.tight_layout()
    plt.show()
    return 


plot_time_and_distance_per_aruco(df_1_with_metrics, col_1)

plot_time_and_distance_per_aruco(df_2_with_metrics, col_2)

In [ ]:
#Optional test 
aruco = df_1_with_metrics[df_1_with_metrics['ARUCO_number'] == 1]

plt.scatter(aruco['X'], aruco['Y'], s =1)
plt.plot(
    [roi_bounds_nest_1['x_min'], roi_bounds_nest_1['x_min'], roi_bounds_nest_1['x_max'], roi_bounds_nest_1['x_max'], roi_bounds_nest_1['x_min']],
    [roi_bounds_nest_1['y_min'], roi_bounds_nest_1['y_max'], roi_bounds_nest_1['y_max'], roi_bounds_nest_1['y_min'], roi_bounds_nest_1['y_min']],
    'g--', label='ROI Boundary'
)
plt.show()

plt.plot(aruco['time_outside_roi'])
plt.show()

In [ ]:
df_1_clean = df_1_with_metrics.drop(columns= [ 'inside_roi', 'outside_roi' ])
df_2_clean = df_2_with_metrics.drop(columns= [ 'inside_roi', 'outside_roi' ])


In [ ]:
df_1_clean

In [ ]:
#Save the active_arucos (uncleaned) 
def convert_to_pickle(folder_path, pickle_name, df_to_convert): 
    #Create a new folder with the combined files in the desired directory 
    folder_name = folder_path.split('/')[-2]
    if not os.path.exists(folder_path):
        # If the folder does not exist, create it
        os.makedirs(folder_path)
        print(f"Folder '{folder_name}' created in '{folder_path}'.")
    else:
        print(f"Folder '{folder_name}' already exists in '{folder_path}'.")

    df_to_convert.to_pickle(folder_path +  pickle_name )
    
    return

convert_to_pickle(output_folder_path, exp_name + '_4_aruco_with_time_outside_nest_col1.pkl', df_1_clean)
convert_to_pickle(output_folder_path, exp_name + '_4_aruco_with_time_outside_nest_col2.pkl', df_2_clean)
